# Feature extraction (spaCy + ELECTRA)

Per essay we get **S** (spaCy writing stats) and **E** (ELECTRA meaning), stick them into **H**, then train four XGBoost setups on a 70/15/15 split. Run the notebook top to bottom. Turn on `RUN_FULL_PIPELINE` only after the smoke cell works.

Order: load table → **S** → **E** (close spaCy first to free the GPU) → **H** → split + text-only **R** → hybrid, spaCy-only, ELECTRA-only, and TF-IDF models.


In [ ]:
# Where data lives, batch sizes, and full run vs a short dry run.
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
COMBINED_CSV = DATA_DIR / "processed" / "combined_dataset.csv"
FEATURES_DIR = DATA_DIR / "processed" / "features"
MODELS_DIR = DATA_DIR / "processed" / "models"

RUN_FULL_PIPELINE = True
SPACY_BATCH_SIZE = 64
ELECTRA_BATCH_SIZE = 16
ELECTRA_CHECKPOINT_EVERY = 500

import spacy

if not spacy.util.is_package("en_core_web_sm"):
    from spacy.cli import download

    download("en_core_web_sm")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Combined CSV exists:", COMBINED_CSV.exists())


## GPU checks

Make sure the GPU actually works before you extract everything. Later cells follow `USE_GPU` from here.

In [ ]:
# Quick check that torch, spaCy, and XGBoost see the GPU.
import spacy
import xgboost as xgb

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

try:
    import cupy
    print("CuPy:", cupy.__version__)
except ImportError as e:
    print("CuPy missing (pip install cupy-cuda12x):", e)

print("XGBoost:", xgb.__version__)

USE_GPU = torch.cuda.is_available()
if USE_GPU:
    spacy.require_gpu()
    print("spaCy GPU: require_gpu OK")
else:
    print("WARNING: No CUDA — run full 157k on a GPU machine.")

rng = np.random.default_rng(0)
x_smoke = rng.random((1000, 20), dtype=np.float32)
y_smoke = (x_smoke[:, 0] > 0.5).astype(int)
clf = xgb.XGBClassifier(
    objective="binary:logistic",
    device="cuda" if USE_GPU else "cpu",
    tree_method="hist",
    n_estimators=10,
    max_depth=3,
    n_jobs=1,
)
clf.fit(x_smoke, y_smoke)
print("XGBoost smoke fit OK on", "cuda" if USE_GPU else "cpu")


## Smoke test

Tiny **S**, **E**, and **H** on three strings (including one empty row). If this fails, don't run the full table.

In [ ]:
from utils.features.spacy_features import (
    SPACY_FEATURE_NAMES,
    extract_spacy_matrix,
    load_spacy_nlp,
)
from utils.features.electra_features import extract_electra_embeddings, load_electra
from utils.features.late_fusion import fuse_spacy_electra

sample_texts = [
    "Academic writing requires clarity and evidence-based argumentation.",
    "The experiment measured reaction time across three conditions.",
    "",
]

nlp = load_spacy_nlp(use_gpu=USE_GPU)
S_smoke = extract_spacy_matrix(sample_texts, nlp, batch_size=8, show_progress=False)
assert S_smoke.shape == (3, len(SPACY_FEATURE_NAMES))
assert not np.isnan(S_smoke).any()
print("S smoke shape:", S_smoke.shape)

tokenizer, electra_model, device = load_electra(prefer_cuda=USE_GPU)
E_smoke = extract_electra_embeddings(
    sample_texts,
    tokenizer,
    electra_model,
    device,
    batch_size=2,
    show_progress=False,
)
assert E_smoke.shape == (3, 768)
assert not np.isnan(E_smoke).any()
print("E smoke shape:", E_smoke.shape)

H_smoke, names = fuse_spacy_electra(S_smoke, E_smoke, SPACY_FEATURE_NAMES)
assert H_smoke.shape[1] == len(SPACY_FEATURE_NAMES) + 768
# Empty essay row should be all zeros in **H** too.
assert H_smoke[2].sum() == 0.0
print("H smoke shape:", H_smoke.shape)
print("Smoke checks passed.")


## Load combined data

Don't reorder rows after this — spaCy and ELECTRA assume the same essay order. `RUN_FULL_PIPELINE=False` keeps 200 rows for a quick try.

In [ ]:
if not COMBINED_CSV.exists():
    raise FileNotFoundError(f"Run data_cleaning.ipynb first: {COMBINED_CSV}")

df = pd.read_csv(COMBINED_CSV)
expected_cols = {"text", "label", "source", "subject"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Quick dry run — flip RUN_FULL_PIPELINE when smoke + GPU checks look good.
if not RUN_FULL_PIPELINE:
    df = df.head(200).copy()
    print("RUN_FULL_PIPELINE=False — using first 200 rows for a dry run.")

print(df.shape)
print(df["label"].value_counts())
df.head(10)


## **S** from spaCy

One stats vector per essay; labels saved beside it for checking. The ELECTRA cell below must use the same essays in the same order.

In [ ]:
from utils.features.spacy_features import save_spacy_features

nlp = load_spacy_nlp(use_gpu=USE_GPU)
S, spacy_parquet = save_spacy_features(
    df["text"].tolist(),
    df["label"].tolist(),
    df["source"].tolist(),
    df["subject"].tolist(),
    FEATURES_DIR,
    nlp,
    batch_size=SPACY_BATCH_SIZE,
)
print("Saved:", spacy_parquet)
print("S:", S.shape)


## **E** from ELECTRA

Close spaCy and clear GPU memory so ELECTRA has room. On a long run we save **E** in chunks so you don't lose progress.

In [ ]:
from utils.features.electra_features import extract_electra_embeddings, load_electra

# Make room on the GPU before loading ELECTRA.
del nlp
if USE_GPU:
    torch.cuda.empty_cache()

tokenizer, electra_model, device = load_electra(prefer_cuda=USE_GPU)
electra_path = FEATURES_DIR / "electra_embeddings.npy"
E = np.array(
    extract_electra_embeddings(
        df["text"].tolist(),
        tokenizer,
        electra_model,
        device,
        batch_size=ELECTRA_BATCH_SIZE,
        checkpoint_path=electra_path,
        checkpoint_every=ELECTRA_CHECKPOINT_EVERY,
    )
)
np.save(electra_path, E)
del electra_model, tokenizer
if USE_GPU:
    torch.cuda.empty_cache()
print("E:", E.shape, E.dtype)


## **H** — spaCy + ELECTRA together

Paste **S** and **E** into one row per essay. We don't rescale the whole table here — that would mix test essays into the scaling.

In [ ]:
from utils.features.late_fusion import fuse_spacy_electra, save_hybrid_features

# Same number of essays as the table — spaCy and ELECTRA must stay in lockstep.
assert S.shape[0] == E.shape[0] == len(df)
H, hybrid_names = fuse_spacy_electra(S, E, SPACY_FEATURE_NAMES)
hybrid_path = save_hybrid_features(H, hybrid_names, FEATURES_DIR)

row_index = pd.DataFrame({"row_id": np.arange(len(df))})
row_index_path = FEATURES_DIR / "row_index.csv"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
row_index.to_csv(row_index_path, index=False)

print("H:", H.shape)
print("Saved:", hybrid_path, row_index_path)
